# HLLSet Core — Interactive Exploration

This notebook experiments with the `hllset-core` crate: HyperLogLog sets with
lattice algebra, Bell State Similarity morphisms, and content-addressing.

**Prerequisites:** `evcxr_jupyter` Rust kernel must be installed.
Run `cargo install evcxr_jupyter && evcxr_jupyter --install` if not yet set up.

In [ ]:
:dep hllset-core = { path = "/home/alexmy/SGS/SGS_lib/fractal_manifold/hllset-next/crates/hllset-core" }
use hllset_core::*;
use hllset_core::core::bss::BSSResult;

println!("✅ hllset-core loaded — M={} registers, {} bits/reg", M, BITS_PER_REG);

---
## 1. Creating HLLSets from Tokens

HLLSets are created by inscribing tokens (byte sequences). Each token is
hashed with MurmurHash3 and the resulting bit position is set in the bitmap.

In [ ]:
// Create from tokens
let a = HLLSet::from_tokens(&["hello", "world", "test"]);
println!("bits set (popcount): {}", a.popcount());
println!("is empty: {}", a.is_empty());

// Empty set
let empty = HLLSet::new();
println!("empty popcount: {}", empty.popcount());
println!("empty cardinality: {}", empty.cardinality());

### Idempotence: Same token sets the same bit
Re-adding the same token doesn't change the HLLSet — bits are idempotent.

In [ ]:
let mut h = HLLSet::new();
h.add_token(b"alice");
let after_one = h.popcount();

h.add_token(b"alice");  // same token again
h.add_token(b"alice");  // and again

assert_eq!(h.popcount(), after_one, "Idempotence: same token → same bits");
println!("✅ Idempotence verified: popcount remains {}", after_one);

// Order independence
let ab = HLLSet::from_tokens(&["a", "b", "c"]);
let ba = HLLSet::from_tokens(&["c", "a", "b"]);
assert_eq!(ab.popcount(), ba.popcount(), "Order-independence: popcounts match");
println!("✅ Order independence verified");

---
## 2. Cardinality Estimation

Uses the **Horvitz-Thompson estimator** for bitmap registers.
For each bit position s (0..31), count registers where bit s is set,
then estimate via f̂_s = −M × ln(1 − c_s/M).

In [ ]:
// Single token → estimated cardinality ≈ 1
let single = HLLSet::from_tokens(&["hello"]);
println!("1 token:  cardinality ≈ {:.1}", single.cardinality());

// 100 unique tokens
let tokens: Vec<String> = (0..100).map(|i| format!("item_{}", i)).collect();
let hundred = HLLSet::from_tokens(&tokens.iter().map(|s| s.as_str()).collect::<Vec<_>>());
let card = hundred.cardinality();
let error_pct = ((card - 100.0).abs() / 100.0) * 100.0;
println!("100 tokens: cardinality ≈ {:.1} (error: {:.1}%)", card, error_pct);

// 1000 unique tokens
let tokens: Vec<String> = (0..1000).map(|i| format!("item_{}", i)).collect();
let thousand = HLLSet::from_tokens(&tokens.iter().map(|s| s.as_str()).collect::<Vec<_>>());
let card = thousand.cardinality();
let error_pct = ((card - 1000.0).abs() / 1000.0) * 100.0;
println!("1K tokens:  cardinality ≈ {:.1} (error: {:.1}%)", card, error_pct);

### Cardinality grows monotonically
Adding tokens can only increase (or keep the same) the cardinality estimate.

In [ ]:
let mut h = HLLSet::new();
let mut prev = 0.0;
let mut monotonic = true;
for i in 0u32..200 {
    h.add_token(&i.to_le_bytes());
    let cur = h.cardinality();
    if cur < prev - 0.001 { monotonic = false; }
    prev = cur;
}
println!("Monotonic? {} (final cardinality: {:.0})", monotonic, prev);

---
## 3. Set Algebra (Lattice Operations)

HLLSets form a bounded distributive lattice under:
- **Join** (∪): union — bitwise OR
- **Meet** (∩): intersection — bitwise AND

These are associative, commutative, and idempotent.

In [ ]:
let a = HLLSet::from_tokens(&["apple", "banana", "cherry"]);
let b = HLLSet::from_tokens(&["banana", "cherry", "date"]);

// Union: elements in either set
let union = a.union(&b);
println!("A ∪ B:  cardinality ≈ {:.1}", union.cardinality());

// Intersection: elements in both
let inter = a.intersection(&b);
println!("A ∩ B:  cardinality ≈ {:.1}", inter.cardinality());

// Difference: elements in A but not B
let diff = a.difference(&b);
println!("A \\ B: cardinality ≈ {:.1}", diff.cardinality());

// Symmetric difference (XOR): exactly one of the two
let xor = a.symmetric_difference(&b);
println!("A ⊕ B:  cardinality ≈ {:.1}", xor.cardinality());

// Jaccard similarity
let jac = a.jaccard_similarity(&b);
println!("Jaccard(A,B): {:.3}", jac);

### Verifying Lattice Properties

In [ ]:
let x = HLLSet::from_tokens(&["a", "b", "c"]);
let y = HLLSet::from_tokens(&["b", "c", "d"]);
let z = HLLSet::from_tokens(&["c", "d", "e"]);

// Idempotence: x ∪ x = x
assert_eq!(x.union(&x).popcount(), x.popcount(), "Union idempotence");
assert_eq!(x.intersection(&x).popcount(), x.popcount(), "Intersection idempotence");
println!("✅ Idempotence:     x∪x=x, x∩x=x");

// Commutativity: x ∪ y = y ∪ x
assert_eq!(x.union(&y).popcount(), y.union(&x).popcount(), "Union commutativity");
assert_eq!(x.intersection(&y).popcount(), y.intersection(&x).popcount(), "Intersection commutativity");
println!("✅ Commutativity:   x∪y=y∪x, x∩y=y∩x");

// Associativity: (x∪y)∪z = x∪(y∪z)
let left_assoc = x.union(&y).union(&z).popcount();
let right_assoc = x.union(&y.union(&z)).popcount();
assert_eq!(left_assoc, right_assoc, "Union associativity");
println!("✅ Associativity:   (x∪y)∪z = x∪(y∪z)");

// Absorption: x ∩ (x ∪ y) = x
let absorb_left = x.intersection(&x.union(&y)).popcount();
assert_eq!(absorb_left, x.popcount(), "Absorption left");
println!("✅ Absorption:      x∩(x∪y) = x");

---
## 4. Content Addressing

Every HLLSet has a deterministic content key: `h:<sha1>` for heterogeneous
(n-gram tokenized) data, `c:<sha1>` for homogeneous (catalog) data.
Same content → same key, always.

In [ ]:
use hllset_core::core::content_addr;

// HLLSet bitmap → content key
let h = HLLSet::from_tokens(&["hello", "world"]);
let key1 = h.content_key();
let hash1 = h.content_hash();
println!("Content key:  {}", key1);
println!("Content hash: {}", hash1);

// Same content → same key (even from different creation)
let h2 = HLLSet::from_tokens(&["hello", "world"]);
assert_eq!(h.content_key(), h2.content_key(), "Deterministic key");
println!("✅ Same tokens → same key");

// Different content → different key
let h3 = HLLSet::from_tokens(&["goodbye"]);
assert_ne!(h.content_key(), h3.content_key(), "Different sets → different keys");
println!("✅ Different sets → different keys");

### Token-based content keys (`h:` prefix)

In [ ]:
// Token-based keys are order-independent and dedup-safe
let k1 = content_addr::content_key_from_tokens(&[b"a", b"b", b"c"]);
let k2 = content_addr::content_key_from_tokens(&[b"c", b"a", b"b"]); // permuted
let k3 = content_addr::content_key_from_tokens(&[b"a", b"a", b"b", b"c"]); // duplicates

assert_eq!(k1, k2, "Order-independence: same set → same key");
assert_eq!(k1, k3, "Dedup: duplicates don't change key");
println!("k1 (a,b,c):    {}", k1);
println!("k2 (c,a,b):    {}", k2);
println!("k3 (a,a,b,c):  {}", k3);
println!("✅ Token keys: order-independent + dedup-safe");

### Catalog keys (`c:` prefix) and Ontological keys

In [ ]:
// Catalog keys for homogeneous (enumerable) data
let values: Vec<&[u8]> = vec![
    b"alice@example.com" as &[u8],
    b"bob@example.com" as &[u8],
    b"carol@example.com" as &[u8],
];
let cat_key = content_addr::content_key_from_catalog(&values);
println!("Catalog key: {}", cat_key);


---
## 5. Bell State Similarity (BSS) Morphisms

BSS is the **directed similarity measure** from HLLSet theory:
- **BSSτ (inclusion):** |A ∩ B| / |B| — how much of B is contained in A?
- **BSSρ (exclusion):** |A \ B| / |B| — how much novelty does A have relative to B?

A **morphism** A → B holds when: BSSτ ≥ τ_min AND BSSρ ≤ ρ_max

In [ ]:
// Scenario: two documents with overlapping vocabulary
let doc_a = HLLSet::from_tokens(&["machine", "learning", "neural", "network", "deep", "gradient"]);
let doc_b = HLLSet::from_tokens(&["neural", "network", "deep", "training", "batch"]);

let tau = doc_a.bss_inclusion(&doc_b);
let rho = doc_a.bss_exclusion(&doc_b);

println!("A = {{machine, learning, neural, network, deep, gradient}}");
println!("B = {{neural, network, deep, training, batch}}");
println!();
println!("BSSτ (A→B) = |A∩B|/|B| = {:.3}", tau);
println!("  → {:.0}% of B's content is also in A", tau * 100.0);
println!();
println!("BSSρ (A→B) = |A\\B|/|B| = {:.3}", rho);
println!("  → A has {:.0}% novelty relative to B's size", rho * 100.0);

In [ ]:
// Morphism check: does A → B hold?
let result = doc_a.morph_to(&doc_b, 0.6, 0.5);
println!("Morphism A → B with τ≥0.6, ρ≤0.5: {}", 
    if result.morphism_holds { "✅ HOLDS" } else { "❌ FAILS" });
println!("  τ = {:.3}, ρ = {:.3}", result.inclusion, result.exclusion);

// Stricter thresholds
let strict = doc_a.morph_to(&doc_b, 0.8, 0.2);
println!("Morphism A → B with τ≥0.8, ρ≤0.2: {}", 
    if strict.morphism_holds { "✅ HOLDS" } else { "❌ FAILS" });

// Self-morphism (always holds at any threshold)
let self_check = doc_a.morph_to(&doc_a, 0.99, 0.01);
println!("Morphism A → A with τ≥0.99, ρ≤0.01: {}", 
    if self_check.morphism_holds { "✅ HOLDS" } else { "❌ FAILS" });

### Morphism Composition
If A → B and B → C, then A → C with (min τ, max ρ).

In [ ]:
let a = HLLSet::from_tokens(&["cat", "dog", "bird", "fish", "hamster"]);
let b = HLLSet::from_tokens(&["cat", "dog", "bird"]);
let c = HLLSet::from_tokens(&["cat", "dog"]);

let ab = a.morph_to(&b, 0.6, 0.4);
let bc = b.morph_to(&c, 0.6, 0.4);
let ac = a.morph_to(&c, 0.6, 0.4);

println!("A → B: τ={:.3}, ρ={:.3}  holds={}", ab.inclusion, ab.exclusion, ab.morphism_holds);
println!("B → C: τ={:.3}, ρ={:.3}  holds={}", bc.inclusion, bc.exclusion, bc.morphism_holds);
println!("A → C: τ={:.3}, ρ={:.3}  holds={}", ac.inclusion, ac.exclusion, ac.morphism_holds);
println!();

// Composition property: τ_ac ≥ min(τ_ab, τ_bc), ρ_ac ≤ max(ρ_ab, ρ_bc)
let min_tau = ab.inclusion.min(bc.inclusion);
let max_rho = ab.exclusion.max(bc.exclusion);
println!("Composition check:");
println!("  τ(AC)={:.3} ≥ min(τ(AB),τ(BC))={:.3} → {}", 
    ac.inclusion, min_tau, ac.inclusion >= min_tau - 0.02);
println!("  ρ(AC)={:.3} ≤ max(ρ(AB),ρ(BC))={:.3} → {}", 
    ac.exclusion, max_rho, ac.exclusion <= max_rho + 0.02);
println!("  (tolerance ±0.02 for HLL estimation noise)");

---
## 6. Multi-Seed Hashing (Homogeneous Catalogs)

For catalog/enumerable data, we use multiple hash seeds (0, 1, 2) for
3-seed cross-validation. Seed 0 is shared with n-gram tokenization
(the G1 compatibility layer).

In [ ]:
use hllset_core::core::hashing::{murmur3_hash, murmur3_hash_seeded};

// Seed 0 = same as unseeded (G1 compatibility layer)
let h0 = murmur3_hash(b"hello");
let h0s = murmur3_hash_seeded(b"hello", 0);
assert_eq!(h0, h0s);
println!("G1 layer: murmur3_hash == murmur3_hash_seeded(seed=0)");
println!("  hash(\"hello\", seed=0) = 0x{:016x}", h0);

// Different seeds → different hashes (for 3-seed cross-validation)
for seed in 0..=3u64 {
    let h = murmur3_hash_seeded(b"alice@example.com", seed);
    println!("  hash(\"alice@example.com\", seed={}) = 0x{:016x}", seed, h);
}

---
## 7. Serialization Roundtrip

HLLSets serialize to compact Roaring bitmap format and can be
deserialized identically.

In [ ]:
let original = HLLSet::from_tokens(&["serialize", "deserialize", "roundtrip", "test"]);
let original_card = original.cardinality();
let original_key = original.content_key();
let original_pc = original.popcount();

// Serialize
let bytes = original.to_bytes();
println!("Serialized size: {} bytes", bytes.len());

// "Store" and "load"
let restored = HLLSet::from_bytes(&bytes).expect("deserialize should succeed");

assert_eq!(restored.popcount(), original_pc, "Same popcount");
assert_eq!(restored.cardinality(), original_card, "Same cardinality");
assert_eq!(restored.content_key(), original_key, "Same content key");

println!("✅ Roundtrip: {} cardinality, key preserved", restored.cardinality() as u64);

---
## 8. Batch Processing: Union of Many Sets

Demonstrates the lattice's power — union/intersection of arbitrary
collections produces the same result regardless of grouping.

In [ ]:
// Create 10 HLLSets with overlapping tokens
let mut sets: Vec<HLLSet> = Vec::new();
for i in 0..10 {
    let tokens: Vec<String> = (i*10..i*10+15).map(|n| format!("token_{}", n)).collect();
    sets.push(HLLSet::from_tokens(&tokens.iter().map(|s| s.as_str()).collect::<Vec<_>>()));
}

// Batch union: all at once vs incremental
let batch_union = HLLSet::union_all(sets.clone());

let mut incremental = HLLSet::new();
for s in &sets {
    incremental.merge(s);
}

assert_eq!(batch_union.popcount(), incremental.popcount(),
    "Batch union = incremental union");
println!("✅ Batch union = incremental union");
println!("   Total cardinality: {:.1} (from {} overlapping sets)", 
    batch_union.cardinality(), sets.len());

// Grouping doesn't matter
let group_a = HLLSet::union_all(sets[..5].to_vec()).union(&HLLSet::union_all(sets[5..].to_vec()));
assert_eq!(group_a.popcount(), batch_union.popcount(), "Grouping independence");
println!("✅ Grouping independence: (∪ first 5) ∪ (∪ last 5) = ∪ all 10");

---
## 9. Approximate Equality

`approx_eq` uses symmetric BSS to check if two HLLSets represent
approximately the same content.

In [ ]:
// Same content → approx equal
let v1 = HLLSet::from_tokens(&["x", "y", "z"]);
let v2 = HLLSet::from_tokens(&["x", "y", "z"]);
println!("Same content:     approx_eq = {}", v1.approx_eq(&v2, 0.9));

// Different content → not approx equal
let v3 = HLLSet::from_tokens(&["a", "b", "c"]);
println!("Different content: approx_eq = {}", v1.approx_eq(&v3, 0.9));

// Mostly overlapping content
let shared: Vec<&str> = (0..30).map(|i| {
    // Just using string literals from a fixed set
    ["alpha", "beta", "gamma", "delta", "epsilon",
     "zeta", "eta", "theta", "iota", "kappa"][i % 10]
}).collect();
let big_a_tokens: Vec<&str> = shared.iter().copied().chain(["unique_a1", "unique_a2"]).collect();
let big_b_tokens: Vec<&str> = shared.iter().copied().chain(["unique_b1"]).collect();

let big_a = HLLSet::from_tokens(&big_a_tokens);
let big_b = HLLSet::from_tokens(&big_b_tokens);
println!("Overlapping sets: approx_eq (τ=0.8) = {}", big_a.approx_eq(&big_b, 0.8));
println!("Overlapping sets: approx_eq (τ=0.95) = {}", big_a.approx_eq(&big_b, 0.95));

---
## 10. Summary: Key Properties Demonstrated

| Property | Verification |
|----------|-------------|
| **Idempotence** | Same token → same bits. x∪x = x, x∩x = x |
| **Commutativity** | x∪y = y∪x, x∩y = y∩x |
| **Associativity** | (x∪y)∪z = x∪(y∪z) |
| **Absorption** | x∩(x∪y) = x |
| **Content addressing** | Same content → deterministic `h:` or `c:` key |
| **Cardinality monotonicity** | Adding tokens never decreases estimate |
| **BSS morphisms** | Directed similarity with tunable τ,ρ thresholds |
| **Serialization** | Lossless roundtrip via Roaring bitmap format |
| **Batch independence** | ∪ of all = (∪ of group1) ∪ (∪ of group2) |